# Bài tập Buổi 7 - Logistic Regression & KNN

## Phần 2: Logistic Regression & KNN với bộ dữ liệu Dry Bean

**Sinh viên thực hiện:** Nguyễn Bá Quốc Long

---


## 0. Chuẩn bị môi trường & Import Thư viện

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


## 1. Tải dữ liệu (đã cho)


In [58]:
df_train = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
df_test = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')
display(df_train)
display(df_test)

,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10829,33838,661.029,222.973514,193.483638,1.152415,0.497014,34146,207.566567,0.782128,0.990980,0.973136,0.930902,0.006589,0.003052,0.866579,0.998659,SEKER
10830,50111,848.651,312.166965,208.195970,1.499390,0.745113,51358,252.593165,0.796323,0.975719,0.874349,0.809160,0.006230,0.001647,0.654741,0.981712,SIRA
10831,38286,725.536,263.207797,185.374667,1.419869,0.709912,38646,220.787792,0.735279,0.990685,0.913970,0.838835,0.006875,0.002100,0.703643,0.999082,SEKER
10832,69509,1046.853,373.829547,238.145900,1.569750,0.770827,70544,297.492197,0.731366,0.985328,0.797039,0.795796,0.005378,0.001331,0.633292,0.994110,BARBUNYA


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,40512,737.636,248.202030,208.525152,1.190274,0.542365,40973,227.115566,0.744268,0.988749,0.935641,0.915043,0.006127,0.002650,0.837304,0.996621,SEKER
1,31890,660.655,250.417300,162.522502,1.540816,0.760783,32240,201.503372,0.801962,0.989144,0.918153,0.804670,0.007853,0.002031,0.647494,0.997670,DERMASON
2,33107,671.869,244.728317,172.938116,1.415121,0.707560,33506,205.312303,0.798683,0.988092,0.921638,0.838940,0.007392,0.002259,0.703820,0.995990,DERMASON
3,61684,984.878,412.115648,191.473280,2.152340,0.885515,62303,280.247227,0.784724,0.990065,0.799130,0.680021,0.006681,0.000881,0.462428,0.995303,HOROZ
4,37189,700.253,243.343401,194.910062,1.248491,0.598708,37534,217.601713,0.784826,0.990808,0.953047,0.894217,0.006543,0.002581,0.799623,0.998322,SEKER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2704,73480,1069.283,413.783300,230.200405,1.797492,0.830961,75618,305.871937,0.771849,0.971726,0.807595,0.739208,0.005631,0.001037,0.546429,0.982201,CALI
2705,80301,1093.430,428.404507,239.918349,1.785626,0.828474,80972,319.753669,0.793833,0.991713,0.844012,0.746383,0.005335,0.001021,0.557087,0.994749,CALI
2706,38747,704.737,245.542069,201.230326,1.220204,0.573029,39008,222.113063,0.791886,0.993309,0.980379,0.904583,0.006337,0.002617,0.818270,0.998456,SEKER
2707,52530,831.059,296.630483,225.918341,1.312999,0.648029,52917,258.618006,0.749572,0.992687,0.955770,0.871852,0.005647,0.002013,0.760127,0.998045,SEKER


## 2. Tách `target` và scale trước khi huấn luyện

In [59]:
X_train = df_train.drop(columns = 'class').copy()
y_train = df_train['class'].copy()
X_test = df_test.drop(columns = 'class').copy()
y_test = df_test['class'].copy()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Logistic Regression

In [60]:
logis_model = LogisticRegression().fit(X_train_scaled, y_train)
logis_pred = logis_model.predict(X_test_scaled)

In [61]:
accuracy_logis = accuracy_score(y_test, logis_pred)
precision_logis = precision_score(y_test, logis_pred, average='weighted')
recall_logis = recall_score(y_test, logis_pred, average='weighted')
f1_logis = f1_score(y_test, logis_pred, average='weighted')
confusion_logis = confusion_matrix(y_test, logis_pred)
print(f'Accuracy: {accuracy_logis:.4f} | Precision: {precision_logis:.4f} | Recall: {recall_logis:.4f} | F1: {f1_logis:.4f}')
print(confusion_logis)

Accuracy: 0.9192 | Precision: 0.9197 | Recall: 0.9192 | F1: 0.9193
[[236   0  18   0   0   4   7]
 [  0 104   0   0   0   0   0]
 [  8   0 307   0   5   2   4]
 [  0   0   0 643   0  13  53]
 [  1   0  11   5 351   0   4]
 [  9   0   0   5   0 383   9]
 [  1   0   1  41   8  10 466]]


## 4. KNN

In [62]:
knn_model = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
knn_predict = knn_model.predict(X_test_scaled)

In [63]:
accuracy_KNN = accuracy_score(y_test, knn_predict)
precision_KNN = precision_score(y_test, knn_predict, average='weighted')
recall_KNN = recall_score(y_test, knn_predict, average='weighted')
f1_KNN = f1_score(y_test, knn_predict, average='weighted')
confusion_KNN = confusion_matrix(y_test, knn_predict)
print(f'Accuracy: {accuracy_KNN:.4f} | Precision: {precision_KNN:.4f} | Recall: {recall_KNN:.4f} | F1: {f1_KNN:.4f}')
print(confusion_KNN)

Accuracy: 0.9155 | Precision: 0.9163 | Recall: 0.9155 | F1: 0.9157
[[232   0  21   0   1   3   8]
 [  0 104   0   0   0   0   0]
 [  8   0 309   0   5   2   2]
 [  0   0   0 647   0  10  52]
 [  0   0  13   4 347   0   8]
 [  6   0   1   7   0 380  12]
 [  3   0   0  49   8   6 461]]


---